# JustBuild Foundation - Customer Requests Agent

Twenty-five customer messages, one response policy, one routing table. This notebook runs the same **three-agent** design you drew on the board, now pointed at the support inbox:

**Input -> Agent 1 (Ingest & Extract) -> Agent 2 (Judge & Prioritise) -> Agent 3 (Communicate) -> You (the human)**

- **Agent 1** reads each message and works out what the customer wants and how they feel.
- **Agent 2** sets the priority, routes to the right team, and attaches the response target. The priority rules and routing live in **code**, not in the model.
- **Agent 3** drafts a reply for the customer. It **sends nothing** - that stays your call.

Everything the notebook needs - the messages, the response policy, and the **OC Brain** - is pulled from the hosted pack. Nothing is pasted inline.

> Run the cells top to bottom. Watch for the green checkmarks.

In [ ]:
#@title 1 · Setup  { display-mode: "form" }
!pip -q install google-generativeai pypdf requests pandas >/dev/null 2>&1
import requests, io, re, json, time
from pypdf import PdfReader
import pandas as pd
print('Setup complete - libraries ready.')

In [ ]:
#@title 2 · Configuration  { display-mode: "form" }
import getpass

MODE = "Live Gemini"  #@param ["Live Gemini", "Switch to OC Brain"]
MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}

BASE = "https://justbuild.orangecaterpillar.com/foundation-material/customer-requests"
MESSAGES_URL = f"{BASE}/CustomerMessages.pdf"
POLICY_URL   = f"{BASE}/Response_Policy.pdf"
OC_BRAIN_URL = f"{BASE}/oc-brain.json"

GEMINI_API_KEY = ""
if MODE == "Live Gemini":
    GEMINI_API_KEY = getpass.getpass('Paste your Gemini API key (hidden). Leave blank to use the OC Brain: ').strip()

print(f'Mode: {MODE}   Model: {MODEL}')

In [ ]:
#@title 3 · Load the messages, the policy, and the OC Brain  { display-mode: "form" }
def fetch_pdf_pages(url):
    r = requests.get(url, timeout=30); r.raise_for_status()
    reader = PdfReader(io.BytesIO(r.content))
    return [(p.extract_text() or '') for p in reader.pages]

message_pages = fetch_pdf_pages(MESSAGES_URL)
MESSAGE_BLOCKS = []
for t in message_pages:
    if re.search(r'Message \d+ of \d+', t):
        MESSAGE_BLOCKS.append(re.sub(r'Message \d+ of \d+\s*', '', t).strip())

POLICY_TEXT = '\n'.join(fetch_pdf_pages(POLICY_URL))
OC_BRAIN = requests.get(OC_BRAIN_URL, timeout=30).json()

# Read the routing table and the SLA targets out of the policy. Fall back to the OC Brain's copy.
ROUTING = {}
for cat, team in re.findall(r'ROUTE:\s*(.+?)\s*\|\s*(.+)', POLICY_TEXT):
    if cat.strip().lower() != 'category':
        ROUTING[cat.strip()] = team.strip()
SLA = {}
for p, target in re.findall(r'SLA:\s*(Urgent|High|Normal|Low)\s*\|\s*(.+)', POLICY_TEXT):
    SLA[p] = target.strip()
if len(ROUTING) < 3:
    ROUTING = {r['category']: r['team'] for r in OC_BRAIN['routing_table']}
if len(SLA) < 3:
    SLA = {r['priority']: r['target'] for r in OC_BRAIN['sla_table']}

print(f'Loaded {len(MESSAGE_BLOCKS)} messages, {len(ROUTING)} routes, {len(SLA)} SLA targets from the hosted pack.')
print(f"OC Brain on standby - {len(OC_BRAIN['messages'])} prepared results loaded.")

In [ ]:
#@title 4 · The engine (and the OC Brain resilience layer)  { display-mode: "form" }
USE_OC_BRAIN = (MODE == 'Switch to OC Brain') or (not GEMINI_API_KEY)
if USE_OC_BRAIN and MODE == 'Live Gemini':
    print('No API key given - running on the OC Brain.')

model = None
if not USE_OC_BRAIN:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel(MODEL)

def call_gemini(prompt, tries=4):
    last = None
    for i in range(tries):
        try:
            return model.generate_content(prompt).text
        except Exception as e:
            last = e; msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg: what = '429 (busy / free-tier limit)'
            elif '503' in msg or 'UNAVAILABLE' in msg:      what = '503 (model overloaded)'
            elif '404' in msg:                              what = '404 (model name)'
            else:                                           what = 'an error'
            wait = 2 ** i
            print(f'Gemini returned {what} - try {i+1}/{tries}. Waiting {wait}s...')
            time.sleep(wait)
    raise last

def extract_json(text):
    text = text.strip()
    text = re.sub(r'^```(?:json)?', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    start = min([i for i in (text.find('['), text.find('{')) if i != -1])
    return json.loads(text[start:])

def switch_to_oc_brain(reason):
    global USE_OC_BRAIN
    print(f'{reason} Switching to the OC Brain to keep going.')
    USE_OC_BRAIN = True

print('Engine ready.', '(OC Brain)' if USE_OC_BRAIN else f'(Live: {MODEL})')

## Agent 1 - Ingest & Extract
Turn 25 messages into one clean table. Live, Gemini reads each message and judges intent and sentiment; on the OC Brain, the same facts come from the prepared file.

In [ ]:
#@title 5 · Agent 1 runs  { display-mode: "form" }
CATEGORIES = list(ROUTING.keys())
records = []  # one row per message, in queue order

if not USE_OC_BRAIN:
    try:
        numbered = '\n\n'.join(f'--- MESSAGE {i+1} ---\n{t}' for i, t in enumerate(MESSAGE_BLOCKS))
        prompt = (
            'You are Agent 1 in a customer support inbox. For EACH message below, read it and return ONLY '
            'a JSON array, in the same order (no prose), with exactly these fields:\n'
            '  reference (string), customer (string),\n'
            '  category (one of: ' + ', '.join(CATEGORIES) + '),\n'
            '  sentiment ("calm", "frustrated", or "angry"),\n'
            '  churn_risk (true if they mention leaving, cancelling, or switching to a competitor),\n'
            '  vip (true if they signal being a high-value, enterprise, or long-standing customer),\n'
            '  legal_or_public (true if they threaten legal action, a regulator, a chargeback, or going public),\n'
            '  time_sensitive (true if they need it by a deadline or event),\n'
            '  is_simple_enquiry (true if it is a simple question or how-to needing no look-up),\n'
            '  summary (a short phrase).\n\n' + numbered
        )
        for d in extract_json(call_gemini(prompt)):
            records.append({'reference': d.get('reference'), 'customer': d.get('customer'),
                            'extract': {k: d.get(k) for k in ('category','sentiment','churn_risk','vip','legal_or_public','time_sensitive','is_simple_enquiry','summary')}})
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')

if USE_OC_BRAIN:
    records = []
    for msg in OC_BRAIN['messages']:
        ex = msg['extract']
        records.append({'reference': ex['reference'], 'customer': ex['customer'],
                        'extract': {k: ex.get(k) for k in ('category','sentiment','churn_risk','vip','legal_or_public','time_sensitive','is_simple_enquiry','summary')}})

df1 = pd.DataFrame([{'Ref': r['reference'], 'Customer': r['customer'], 'Category': r['extract']['category'],
                     'Sentiment': r['extract']['sentiment'], 'Churn': r['extract']['churn_risk'],
                     'VIP': r['extract']['vip'], 'Legal/Public': r['extract']['legal_or_public']} for r in records])
print(f'Agent 1 extracted {len(df1)} messages:')
df1

## Agent 2 - Judge & Prioritise  (rules in code)
Set priority, route to a team, attach the response target, and order the queue. The priority rules, the routing, and the escalation **hard rule** all live in **code** - the policy decides who is handled first, not the model's mood.

In [ ]:
#@title 6 · Agent 2 runs  { display-mode: "form" }
RANK = {'Urgent': 1, 'High': 2, 'Normal': 3, 'Low': 4}

def priority_of(x):
    if x.get('legal_or_public'): return 'Urgent'                        # hard rule
    if x.get('churn_risk') and x.get('vip'): return 'Urgent'
    if x.get('time_sensitive') and x.get('vip'): return 'Urgent'
    if x.get('churn_risk'): return 'High'
    if x.get('sentiment') == 'angry': return 'High'
    if x.get('vip'): return 'High'
    if x.get('time_sensitive'): return 'High'
    if x.get('is_simple_enquiry'): return 'Low'
    return 'Normal'
def resolve_category(cat):
    if cat in ROUTING: return cat
    cl = (cat or '').lower()
    for k in ROUTING:
        if k.lower() in cl or cl in k.lower() or k.split()[0].lower() in cl: return k
    return None
def team_of(x):
    if x.get('legal_or_public'): return 'Escalations'                   # hard rule
    return ROUTING.get(resolve_category(x.get('category')), 'Frontline Support')
def evidence_of(x):
    if x.get('legal_or_public'): return 'Legal or public threat; escalated to Urgent and routed to Escalations.'
    if x.get('churn_risk') and x.get('vip'): return 'VIP customer at risk of leaving.'
    if x.get('time_sensitive') and x.get('vip'): return 'VIP with a hard deadline.'
    if x.get('churn_risk'): return 'Customer at risk of leaving.'
    if x.get('sentiment') == 'angry': return 'Angry customer; needs a fast, careful reply.'
    if x.get('vip'): return 'High-value customer.'
    if x.get('time_sensitive'): return 'Time-sensitive request.'
    if x.get('is_simple_enquiry'): return 'Simple enquiry; a templated answer will do.'
    return 'Standard request.'

for r in records:
    ex = r['extract']
    r['priority'] = priority_of(ex); r['team'] = team_of(ex)
    r['sla'] = SLA.get(r['priority'], ''); r['evidence'] = evidence_of(ex)

# prioritise the queue: worst first, VIPs and churn risks ahead within a priority
queue = sorted(records, key=lambda r: (RANK[r['priority']], -(1 if r['extract'].get('vip') else 0),
                                       -(1 if r['extract'].get('churn_risk') else 0)))

show = pd.DataFrame([{'#': i+1, 'Ref': r['reference'], 'Priority': r['priority'], 'Team': r['team'],
                      'SLA': r['sla'], 'Why': r['evidence']} for i, r in enumerate(queue)])
from collections import Counter
counts = Counter(r['priority'] for r in queue)
print('Agent 2 triaged the inbox (rules applied in code):', {k: counts[k] for k in ['Urgent','High','Normal','Low'] if k in counts})
show

In [ ]:
#@title    ...the Urgent ones, front and centre  { display-mode: "form" }
URGENT = [r for r in queue if r['priority'] == 'Urgent']
print(f'HANDLE NOW - {len(URGENT)} Urgent message(s):\n')
for r in URGENT:
    print(f"  {r['reference']}  {r['customer']}  ->  {r['team']}  [{r['sla']}]")
    print(f"     {r['evidence']}\n")

## Agent 3 - Communicate  (behind a human gate)
Draft a reply for each Urgent customer and a handover for the rest. The agent **drafts**; it never sends. You approve and send.

In [ ]:
#@title 7 · Agent 3 drafts (nothing is sent)  { display-mode: "form" }
draft = None
if not USE_OC_BRAIN:
    try:
        urgent = [{'reference': r['reference'], 'customer': r['customer'], 'team': r['team'],
                   'sla': r['sla'], 'summary': r['extract'].get('summary'), 'why': r['evidence']} for r in URGENT]
        rest = [{'reference': r['reference'], 'customer': r['customer'], 'priority': r['priority'],
                 'team': r['team'], 'sla': r['sla']} for r in queue if r['priority'] != 'Urgent']
        prompt = (
            'You are Agent 3 in a customer support inbox. Write a short handover note to the support lead. '
            'For each Urgent message, include a brief, empathetic suggested reply to the customer (no promises '
            'you cannot keep) and the team it is routed to. Then summarise the rest of the queue by priority. '
            'Make clear that nothing has been sent and that sending is theirs to approve. Return only the note.\n\n'
            + json.dumps({'urgent': urgent, 'rest': rest})
        )
        draft = call_gemini(prompt)
    except Exception as e:
        switch_to_oc_brain(f'The live model did not come through ({str(e)[:60]}).')
if USE_OC_BRAIN or draft is None:
    draft = OC_BRAIN['draft']

print('=' * 70)
print(draft)
print('=' * 70)
print('\nHUMAN GATE: these are drafts. The agent has sent nothing. You decide what goes out.')

In [ ]:
#@title 8 · Your decision  { display-mode: "form" }
DECISION = "Hold"  #@param ["Hold", "Approve the drafts (I will send them myself)", "Reject"]
if DECISION.startswith('Approve'):
    print('You approved the drafts. The agent still sends nothing - you send them yourself, deliberately.')
elif DECISION == 'Reject':
    print('Rejected. Nothing leaves this notebook.')
else:
    print('On hold. Nothing is sent. The decision stays with you.')

## Keep the result
Save the triaged inbox and the drafts to files you can download or paste into your notes, and compare the agent's queue with the one the room worked out by hand.

In [ ]:
#@title 9 · Save the triaged inbox  { display-mode: "form" }
lines = ['# Triaged inbox - Customer Requests', '',
         f'Mode: {"OC Brain" if USE_OC_BRAIN else MODEL}', '', '## Queue (worst first)', '']
for i, r in enumerate(queue, 1):
    lines.append(f"{i}. **{r['reference']}** [{r['priority']}] - {r['customer']} -> {r['team']} ({r['sla']}) - {r['evidence']}")
lines += ['', '## Handover and drafted replies', '', '```', draft, '```']
md_out = '\n'.join(lines)

with open('customer_queue.md', 'w') as f: f.write(md_out)
with open('customer_results.json', 'w') as f:
    json.dump({'mode': 'oc_brain' if USE_OC_BRAIN else MODEL,
               'queue': [{'reference': r['reference'], 'priority': r['priority'], 'team': r['team'],
                          'sla': r['sla'], 'why': r['evidence']} for r in queue],
               'draft': draft}, f, indent=2)

print('Saved: customer_queue.md and customer_results.json (open them from the file panel on the left).')
print('\n----- copy from here -----\n')
print(md_out)
# To download instead, uncomment:
# from google.colab import files; files.download('customer_queue.md')

## What you just saw
The same three narrow agents, chained: extract, judge, communicate - the judge is pure **code** again, so the customers who need us most are always handled first, and the final call still belongs to a **human**. Swap this pack for your own and the pipeline serves a different job, which is exactly what you'll do next with your own use-case.